Cell 1 — Install libraries


In [1]:
!pip install transformers datasets evaluate gradio -q
!pip install accelerate -U -q

# torchvision is not needed for this text-only task — removing it avoids
# a known bug where 'datasets' tries to import VideoReader from torchvision.io
!pip uninstall -y torchvision -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00


Cell 3 — Imports


In [1]:
import torch  # Core PyTorch library, used for tensors and GPU handling
from datasets import load_dataset  # Loads datasets from Hugging Face Hub
from transformers import (
    BertTokenizerFast,              # Fast tokenizer for BERT
    BertForSequenceClassification,  # BERT model with classification head
    TrainingArguments,              # Training configuration class
    Trainer                         # High-level training loop
)
import numpy as np   # Numerical operations on predictions
import evaluate       # Metric library (accuracy, f1, etc.)

Cell 4 — Sanity check (GPU + torchvision fix)


In [2]:
print("GPU available:", torch.cuda.is_available())  # should print True

import datasets.formatting.torch_formatter as tf
print("Torchvision bug avoided:", not tf.config.TORCHVISION_AVAILABLE)  # should print True

GPU available: True
Torchvision bug avoided: True


Cell 5 — Load the AG News dataset


In [6]:
# Downloads the official AG News dataset (train + test split) from HF Hub
dataset = load_dataset("aRWA787/ag_news_dataset")

print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_name', 'length'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'label_name', 'length'],
        num_rows: 7600
    })
})
{'text': "wall st. bears claw back into the black (reuters) reuters - short-sellers, wall street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2, 'label_name': 'Business', 'length': 144}


Cell 6 — Tokenizer + preprocessing


In [7]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Cell 7 — Train/test subsets


In [9]:
train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(5000))
test_dataset = tokenized_dataset["test"].shuffle(seed=42).select(range(1000))

Cell 8 — Load model


In [10]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Cell 9 — Metrics function


In [11]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

Cell 10 — Training arguments


In [13]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

Cell 11 — Train


In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.339750,0.337680,0.890000,0.890166
2,0.176360,0.315620,0.900000,0.900317
3,0.096887,0.323915,0.909000,0.909242


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=939, training_loss=0.26403245580589935, metrics={'train_runtime': 397.9883, 'train_samples_per_second': 37.69, 'train_steps_per_second': 2.359, 'total_flos': 986684175360000.0, 'train_loss': 0.26403245580589935, 'epoch': 3.0})

 Evaluate:

In [15]:
results = trainer.evaluate()
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.096887,0.323915,3,0.909000,0.909242


{'eval_loss': 0.3239152133464813, 'eval_accuracy': 0.909, 'eval_f1': 0.9092421142596322}


 Save the model:

In [16]:
model.save_pretrained("./news_classifier_model")
tokenizer.save_pretrained("./news_classifier_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./news_classifier_model/tokenizer_config.json',
 './news_classifier_model/tokenizer.json')